In [1]:
import matplotlib.pyplot as plt
import numpy as np

### Soft margin classifier

In [2]:
from sklearn.datasets import load_iris
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

In [3]:
iris = load_iris(as_frame=True)

In [4]:
iris.feature_names

['sepal length (cm)',
 'sepal width (cm)',
 'petal length (cm)',
 'petal width (cm)']

In [5]:
X = iris.data[['petal length (cm)','petal width (cm)']].values
y = (iris.target==2) # Iris virginica
linear_svc = make_pipeline(
    StandardScaler(),
    LinearSVC(C=1,random_state=42) 
)
linear_svc.fit(X,y)

,steps,"[('standardscaler', ...), ('linearsvc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001


Small C → allow mistakes → wider margin → more regularization \
Large C → fewer mistakes → narrower margin → less regularization \
C is the inverse of regularization strength : Small C → strong regularization, Large C → weak regularization

In [6]:
X_new = [[5.5, 1.7], [5.0, 1.5]]
linear_svc.predict(X_new)

array([ True, False])

In [7]:
linear_svc.decision_function(X_new)

array([ 0.66163816, -0.22035761])

In [8]:
# grid search cv for hyperparameter tunning
from sklearn.model_selection import GridSearchCV

In [9]:
linear_svc = make_pipeline(
    StandardScaler(),
    LinearSVC() 
)

In [10]:
param_grid = {
    'linearsvc__C':[0.01,0.1,1,10,100]
}

In [11]:
grid_search = GridSearchCV(linear_svc,param_grid,cv=5,scoring="accuracy") # also use scoring="f1" becauese it is calssification

In [12]:
grid_search.fit(X,y)

,estimator,Pipeline(step...LinearSVC())])
,param_grid,"{'linearsvc__C': [0.01, 0.1, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [13]:
grid_search.best_params_

{'linearsvc__C': 0.1}

In [14]:
grid_search.best_score_

np.float64(0.9666666666666668)

In [15]:
grid_search = GridSearchCV(linear_svc,param_grid,cv=5,scoring="f1")
grid_search.fit(X,y)
grid_search.best_params_,grid_search.best_score_

({'linearsvc__C': 0.1}, np.float64(0.9512417407154249))

In [16]:
best_model = grid_search.best_estimator_
best_model.predict(X_new)

array([ True, False])

RandomizedSearchCV tries random combinations of hyperparameters instead of trying all possible combinations.You decide how many trials to run.\
Use RandomizedSearchCV when:\
You have many hyperparameters\
Hyperparameters are continuous (like C)\
GridSearch would be too slow\
You want a good solution quickly, not a perfect one\
In real projects, Random Search is often preferred.

In [17]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform # log-uniform sampling

In [18]:
param_distributions = {
    "linearsvc__C": loguniform(1e-3, 1e3)
}

In [19]:
random_search = RandomizedSearchCV(
    linear_svc,
    param_distributions=param_distributions,
    n_iter=50,
    cv=5,
    scoring="accuracy",
    random_state=42,
)

In [20]:
random_search.fit(X,y)

,estimator,Pipeline(step...LinearSVC())])
,param_distributions,{'linearsvc__C': <scipy.stats....x75b2298bfca0>}
,n_iter,50
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [21]:
random_search.best_params_

{'linearsvc__C': np.float64(0.06690421166498801)}

In [22]:
random_search.best_score_

np.float64(0.9666666666666668)

In [23]:
best_model = random_search.best_estimator_
best_model.predict(X_new)

array([ True, False])

### Nonlinear SVM Classification

In [24]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import PolynomialFeatures

In [25]:
X, y = make_moons(n_samples=100, noise=0.15, random_state=42)

In [28]:
svm_clf = make_pipeline(
    StandardScaler(),
    LinearSVC(C=10,max_iter=10_000,random_state=42)
)

In [29]:
svm_clf.fit(X,y)

,steps,"[('standardscaler', ...), ('linearsvc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001


In [31]:
svm_clf.score(X,y)

0.86

In [26]:
polynomial_svm_clf = make_pipeline(
    PolynomialFeatures(degree=3),
    StandardScaler(),
    LinearSVC(C=10,max_iter=10_000,random_state=42)
)

In [27]:
polynomial_svm_clf.fit(X,y)

,steps,"[('polynomialfeatures', ...), ('standardscaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,degree,3
,interaction_only,False
,include_bias,True
,order,'C'
,copy,True
,with_mean,True
,with_std,True


In [32]:
polynomial_svm_clf.score(X,y)

0.99

### Polynomial Kernel

In [33]:
from sklearn.svm import SVC

In [38]:
svm_clf = make_pipeline(
    StandardScaler(),
    SVC(C=10 ,kernel="poly",degree=3,coef0=1)
)

The coef0 parameter in a SVC model is the independent term (bias term) in the kernel function. It is only significant and used when the kernel is set to 'poly' (polynomial) or 'sigmoid'. Polynomial kernel: (gamma * dot(x, y) + coef0)^degree, Sigmoid kernel: tanh(gamma * dot(x, y) + coef0).  It is ignored if the kernel is 'linear' or 'rbf'. The coef0 controls
how much the model is influenced by high-degree terms versus low-degree
terms.

In [39]:
svm_clf.fit(X,y)

,steps,"[('standardscaler', ...), ('svc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,10
,kernel,'poly'
,degree,3
,gamma,'scale'


In [40]:
svm_clf.score(X,y)

0.99

### Gaussian RBF Kernel

In [41]:
rbf_svm_clf = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf",gamma=5,C=0.001)
)

In [42]:
rbf_svm_clf.fit(X,y)

,steps,"[('standardscaler', ...), ('svc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,0.001
,kernel,'rbf'
,degree,3
,gamma,5


In [43]:
rbf_svm_clf.score(X,y)

0.97

if your model is overfitting, you should reduce γ; if it is underfitting, you should increase γ (similar to the C hyperparameter)

### SVM Regression

In [45]:
from sklearn.svm import LinearSVR

In [44]:
# generate a simple linear dataset
np.random.seed(42)
X = 2 * np.random.rand(50, 1)
y = 4 + 3 * X[:, 0] + np.random.randn(50)

In [53]:
svm_reg = make_pipeline(
    StandardScaler(),
    LinearSVR(epsilon=0.5,random_state=42)
)
svm_reg.fit(X,y)

,steps,"[('standardscaler', ...), ('linearsvr', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,epsilon,0.5
,tol,0.0001
,C,1.0
,loss,'epsilon_insensitive'


In [49]:
svm_reg.score(X,y)

0.7627947445767349

In [51]:
# generate a simple quadratic dataset
np.random.seed(42)
X = 2 * np.random.rand(50, 1) - 1
y = 0.2 + 0.1 * X[:, 0] + 0.5 * X[:, 0] ** 2 + np.random.randn(50) / 10

In [54]:
svm_reg.fit(X,y)
svm_reg.score(X,y)

-0.2681633338260525

In [58]:
# very bad model

In [52]:
from sklearn.svm import SVR

In [55]:
svm_reg = make_pipeline(
    StandardScaler(),
    SVR(kernel="poly",degree=2,C=0.01, epsilon=0.1)
)
svm_reg.fit(X,y)

,steps,"[('standardscaler', ...), ('svr', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,kernel,'poly'
,degree,2
,gamma,'scale'
,coef0,0.0


In [56]:
svm_reg.score(X,y)

0.6663439254285933